In [ ]:
import openai
import httpx
import json
import time
from datetime import datetime, timedelta
from typing import Optional, Dict, Any
from IPython.display import clear_output

# Адреса сервисов
LLM_BASE_URL = "http://llm-vulcan:8080/v1"
LLM_API_KEY = "not-needed"
MOEX_BOT_API = "http://moexapp:80/api/v1"

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="http://llm-vulcan:8080/v1",
    api_key="not-needed",
    model="/models/Qwen3.5-9B-Q4_0.gguf",
    temperature=0.1,
    max_tokens=32768,
    top_p=0.95,
    presence_penalty=1.5,
    extra_body={
        "top_k": 20,
        "chat_template_kwargs": {"enable_thinking": True},
    },
)

# Утилиты с настраиваемым таймаутом
def ask_llm(prompt: str) -> str:
    response = llm.invoke(prompt)
    return response

def api_get(path: str, params: Optional[dict] = None, timeout: int = 120) -> dict:
    """GET-запрос с настраиваемым таймаутом (по умолчанию 120с)."""
    url = f"{MOEX_BOT_API}{path}"
    resp = httpx.get(url, params=params, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def api_post(path: str, json_data: dict, timeout: int = 120) -> dict:
    """POST-запрос с настраиваемым таймаутом."""
    url = f"{MOEX_BOT_API}{path}"
    resp = httpx.post(url, json=json_data, timeout=timeout)
    resp.raise_for_status()
    return resp.json()

def api_delete(path: str, timeout: int = 120) -> None:
    url = f"{MOEX_BOT_API}{path}"
    resp = httpx.delete(url, timeout=timeout)
    resp.raise_for_status()

print("✅ Клиенты и утилиты готовы")

In [ ]:
health = api_get("/system/health")
print("Статус системы:", health["status"])
print("ClickHouse доступен:", health["clickhouse"])
print("Модель загружена:", health["model_loaded"])

metrics = api_get("/system/metrics")
print("Активных задач:", metrics["active_tasks"])
print("Всего задач:", metrics["total_tasks"])
print("Всего свечей в БД (raw_candles):", metrics["total_candles"])

In [ ]:
ticker = "GAZP"
interval = "10m"

task = api_post("/tasks", {"ticker": ticker, "interval": interval})
task_id = task["task_id"]
print(f"Задача создана: {task_id} ({ticker}_{interval}), статус: {task['status']}")

# Ожидание завершения первой загрузки истории и расчёта признаков
print("Ожидание завершения первоначальной загрузки истории и вычислений...")
print("(Может занять 5–30 минут, используется большой таймаут)")

task_info = None
while True:
    # Используем длительный таймаут в начале, затем можно уменьшить
    task_info = api_get(f"/tasks/{task_id}", timeout=300)
    status = task_info.get("status")
    error = task_info.get("error_message")
    last_run = task_info.get("last_run")

    clear_output(wait=True)
    print(f"Статус задачи: {status}")
    if last_run:
        print(f"Последний успешный цикл: {last_run}")
    if error:
        print(f"Ошибка: {error}")

    if status == "running" and last_run:
        print("✅ Задача активна, первый цикл прошёл успешно!")
        break
    if status in ("stopped", "error"):
        print(f"❌ Задача завершилась с ошибкой: {error}")
        break

    time.sleep(5)

In [ ]:
candles = api_get(f"/data/candles/{ticker}", {"interval": interval, "limit": 5})
print(f"Свечей получено: {len(candles.get('candles', []))}")
for c in candles.get("candles", []):
    print(f"  {c['begin']}: O={c['open']}, H={c['high']}, L={c['low']}, C={c['close']}, V={c['volume']}")

# Последнее предсказание
try:
    pred = api_get(f"/data/predictions/{ticker}", {"interval": interval})
    print(f"\nПоследнее предсказание: {pred['value']:.2f} (время {pred['timestamp']})")
except Exception as e:
    print(f"Предсказаний пока нет: {e}")

In [ ]:
# Часовые агрегаты (используется новый эндпоинт)
try:
    agg = api_get(f"/data/aggregates/{ticker}", {"interval": interval, "granularity": "hourly", "limit": 3})
    print("Часовые агрегаты:")
    for a in agg:
        print(f"  {a['timestamp']}: avg={a['avg_price']:.2f}, max={a['max_price']}, min={a['min_price']}, vol={a['total_volume']}")
except Exception as e:
    print(f"Агрегаты не доступны: {e}")

# Сигналы SMA (теперь из таблицы features)
try:
    signals = api_get(f"/data/signals/{ticker}", {"interval": interval, "limit": 3})
    print("\nСигналы SMA:")
    for s in signals:
        print(f"  {s['begin']}: buy={s['buy_signal']}, sell={s['sell_signal']}")
except Exception as e:
    print(f"Сигналы не доступны: {e}")

In [ ]:
# Мониторинг качества модели
try:
    quality = api_get("/monitoring/model_quality", {"ticker": ticker, "interval": interval, "window": 10})
    print("Метрики качества модели:")
    print(f"  RMSE: {quality['metrics']['rmse']:.4f}")
    print(f"  MAPE: {quality['metrics']['mape']:.2f}%")
    print(f"  MAE:  {quality['metrics']['mae']:.4f}")
    print(f"  Статус: {quality['status']}")
    print(f"  Последняя оценка: {quality['last_evaluation_time']}")
except Exception as e:
    print(f"Метрики ещё не готовы: {e}")

# История качества
try:
    hist = api_get("/monitoring/model_quality/history", {"ticker": ticker, "interval": interval, "limit": 5})
    print("\nИстория метрик (последние 5 окон):")
    for h in hist["history"]:
        print(f"  {h['last_evaluation_time']}: RMSE={h['metrics']['rmse']:.4f}, MAPE={h['metrics']['mape']:.2f}%")
except Exception as e:
    print(f"История не доступна: {e}")

In [ ]:
# Экспорт свечей в CSV
try:
    resp = httpx.get(f"{MOEX_BOT_API}/export/candles/{ticker}", params={"interval": interval, "format": "csv"})
    resp.raise_for_status()
    lines = resp.text.strip().split('\n')
    print("Первые 5 строк экспорта CSV:")
    for line in lines[:6]:
        print(line)
except Exception as e:
    print(f"Экспорт не удался: {e}")

In [ ]:
# Работа с агентским SQL-эндпоинтом (read-only)
# Проверим количество свечей и последнюю дату
sql_query = f"SELECT count() AS cnt, max(begin) AS last_date FROM raw_candles_{ticker}_{interval}"
try:
    sql_result = api_post("/sql/query", {"query": sql_query})
    print("Результат SQL:")
    print(f"  Колонки: {sql_result['columns']}")
    for row in sql_result['rows']:
        print(f"  {row}")
except Exception as e:
    print(f"SQL запрос не выполнен: {e}")

In [ ]:
# Корреляции между GAZP и SBER (если есть задача для SBER)
# Для демонстрации сначала запустим задачу для SBER
print("Запуск задачи для SBER 10m...")
sber_task = api_post("/tasks", {"ticker": "SBER", "interval": "10m"})
sber_task_id = sber_task["task_id"]
print(f"SBER task: {sber_task_id}")

# Ждём, пока SBER пройдёт первый цикл (увеличим таймаут)
print("Ожидание первого цикла SBER...")
while True:
    info = api_get(f"/tasks/{sber_task_id}", timeout=300)
    if info["status"] == "running" and info.get("last_run"):
        break
    if info["status"] in ("stopped", "error"):
        print(f"SBER задача завершилась: {info['error_message']}")
        break
    time.sleep(5)

try:
    corr_result = api_get("/data/correlations", {"tickers": "GAZP,SBER", "interval": "10m", "window": 100})
    print("\nКорреляции GAZP-SBER:")
    for c in corr_result["correlations"]:
        print(f"  {c['ticker1']} vs {c['ticker2']}: {c['correlation']}")
except Exception as e:
    print(f"Корреляции не рассчитаны: {e}")

In [ ]:
# Взаимодействие с LLM-агентом (аналитический отчёт)
try:
    candles = api_get(f"/data/candles/{ticker}", {"interval": interval, "limit": 3})
    pred = api_get(f"/data/predictions/{ticker}", {"interval": interval})
    signals = api_get(f"/data/signals/{ticker}", {"interval": interval, "limit": 1})
    quality = api_get("/monitoring/model_quality", {"ticker": ticker, "interval": interval, "window": 10})

    summary = f"""
Тикер: {ticker}
Интервал: {interval}
Последние свечи (цена закрытия): {', '.join([f"{c['begin']}: {c['close']}" for c in candles.get('candles', [])])}
Последнее предсказание модели: {pred['value']:.2f} (на {pred['timestamp']})
Текущие метрики модели: RMSE={quality['metrics']['rmse']:.4f}, MAPE={quality['metrics']['mape']:.2f}%
Последний сигнал SMA: buy={signals[0]['buy_signal']}, sell={signals[0]['sell_signal']}
"""
    prompt = (
        "Ты — старший финансовый аналитик. На основе предоставленных данных по акциям GAZP "
        "дай краткую оценку текущей ситуации: тренд, волатильность, надёжность прогноза. "
        "Также предложи возможное действие (покупка/продажа/ожидание) с аргументацией.\n\n"
        "Данные:\n" + summary
    )
    agent_answer = ask_llm(prompt)
    print("Ответ агента:\n", agent_answer)
except Exception as e:
    print(f"Не удалось собрать данные для агента: {e}")

In [ ]:
# Остановка и перезапуск задачи
print("Остановка задачи...")
api_delete(f"/tasks/{task_id}")
time.sleep(2)
info = api_get(f"/tasks/{task_id}")
print(f"Статус: {info['status']}")

print("Перезапуск задачи...")
new_task = api_post(f"/tasks/{task_id}/restart", {})
print(f"Новая задача: {new_task['task_id']}, статус: {new_task['status']}")

# Дадим немного поработать, затем остановим
time.sleep(10)
api_delete(f"/tasks/{new_task['task_id']}")
print("Задача остановлена.")